# 05 - Model Evaluation

This notebook evaluates the classifiers trained on the DDXPlus-derived feature matrix and shows presentation-ready metrics, confusion matrices, and differential diagnosis examples.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import joblib

sns.set_style("whitegrid")

# Find repository root
repo_root = Path(r"c:\healix-ai-medical-assistant")
X_test = pd.read_csv(repo_root / "training" / "data" / "processed" / "X_test.csv")
y_test = pd.read_csv(repo_root / "training" / "data" / "processed" / "y_test.csv")["target"]

model_path = repo_root / "training" / "models" / "xgboost_model.pkl"
model = joblib.load(model_path)
label_encoder = joblib.load(repo_root / "training" / "models" / "label_encoder.joblib")

preds = model.predict(X_test)
print(classification_report(y_test, preds, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm[:20, :20], annot=False, cmap="Blues")
plt.title("Confusion matrix (first 20 classes)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
# ROC-AUC for multi-class using one-vs-rest
classes = label_encoder.classes_
try:
    y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))
    proba = model.predict_proba(X_test)
    macro_auc = roc_auc_score(y_test_bin, proba, average="macro")
    print("Macro ROC-AUC:", round(macro_auc, 4))
except Exception as exc:
    print("ROC-AUC unavailable for this configuration:", exc)

In [ ]:
# Top predicted diseases for real DDXPlus feature rows
sample_indices = [0, 1, 2]
for index in sample_indices:
    sample_vector = X_test.iloc[index].to_frame().T
    active_evidence = [col for col in sample_vector.columns if col.startswith("E_") and int(sample_vector.iloc[0][col]) == 1]
    proba = model.predict_proba(sample_vector)[0]
    ranking = sorted(zip(label_encoder.inverse_transform(np.arange(len(proba))), proba), key=lambda item: item[1], reverse=True)[:5]
    print(f"Example case {index}")
    print("Active evidence:", active_evidence[:15])
    for disease, probability in ranking:
        print(f"- {disease}: {probability:.3f}")
    print("---")

In [ ]:
# Error analysis
errors = pd.DataFrame({"true": y_test, "pred": preds})
errors["is_error"] = errors["true"] != errors["pred"]
error_rows = errors[errors["is_error"]].head(10)
print(error_rows)

## Interpretation

- The classification report shows overall performance and per-class behavior on the DDXPlus-derived feature matrix.
- The confusion matrix highlights where the model confuses similar conditions.
- The differential diagnosis ranking shows probability-based outputs suitable for a clinical decision support system.
- The output is explicitly framed as a differential diagnosis / possible conditions list, not a final diagnosis.